In [1]:
# Load the documents

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('data/subtitles', glob="*.srt", show_progress=True, loader_cls=TextLoader)

kb_docs = loader.load()

100%|█████████████████████████████████████████| 10/10 [00:00<00:00, 7371.36it/s]


In [2]:
# Chunk the loaded documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(kb_docs)

In [3]:
print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 10

Number of Chunks: 514


In [4]:
import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, OpenAIEmbeddingFunction

# Setup API Key
with open('keys/.chroma_api_key.txt') as f:
    CHROMA_API_KEY = f.read()

with open('keys/.chroma_tenant.txt') as f:
    CHROMA_TENANT = f.read()

f = open("keys/.openai_api_key.txt")
OPENAI_API_KEY = f.read()

client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant=CHROMA_TENANT,
  database='chroma_db_1'
)

schema = Schema()
    
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

# # Configure vector index with custom embedding function
# openai_ef = OpenAIEmbeddingFunction(
#     api_key=OPENAI_API_KEY,
#     model_name="text-embedding-3-small"
# )

# schema = schema.create_index(
#     config=VectorIndexConfig(
#         space="cosine",
#         embedding_function=openai_ef
#     )
# )

collection = client.get_or_create_collection(
  name="friends_collection",
  schema=schema,
)

collection.count()

0

In [8]:
chunks[0]

Document(metadata={'source': 'data/subtitles/Friends_2x01.srt'}, page_content='1\n00:00:01,435 --> 00:00:04,082\nThis is pretty much\nwhat\'s happened so far.\n\n2\n00:00:04,395 --> 00:00:07,179\nRoss was in love\nwith Rachel since forever.\n\n3\n00:00:07,423 --> 00:00:10,437\nEvery time he tried to tell her,\nsomething got in the way...\n\n4\n00:00:10,651 --> 00:00:12,529\n...Iike cats, Italian guys.\n\n5\n00:00:12,736 --> 00:00:15,922\nAnd finally, Chandler was,\nlike, "Forget about her."\n\n6\n00:00:16,166 --> 00:00:20,762\nWhen Ross was in China, Chandler\nlet it slip that Ross loved Rachel.')

In [9]:
# import uuid

# collection.add(
#     documents=[chunk.page_content for chunk in chunks],
#     metadatas=[chunk.metadata for chunk in chunks],
#     ids=[str(uuid.uuid4()) for _ in chunks]
# )

In [10]:
import uuid

# Free tier in chroma supports 300 docs to be inserted at a time
BATCH_SIZE = 250

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]

    collection.add(
        documents=[chunk.page_content for chunk in batch],
        metadatas=[chunk.metadata for chunk in batch],
        ids=[str(uuid.uuid4()) for _ in batch]
    )

    print(f"Inserted {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)}")

Inserted 250/514
Inserted 500/514
Inserted 514/514


In [11]:
collection.count()

514

In [13]:
client.list_collections()

[Collection(name=food-collection), Collection(name=friends_collection)]

## **Building an End-to-End RAG Chain**

**Step 1: Initialize the Chroma DB Connection**  
**Step 2: Create a Retriever Object**   
**Step 3: Initialize a Chat Prompt Template**  
**Step 4: Initialize a Generator (i.e. Chat Model)**  
**Step 5: Initialize a Output Parser**   
**Step 6: Define a RAG Chain**  
**Step 7: Invoke the Chain**

In [12]:
# # Initialize a ChromaDB Connection
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings

# f = open("keys/.openai_api_key.txt")
# OPENAI_API_KEY = f.read()
# embedding_model = OpenAIEmbeddings(api_key=OPENAI_API_KEY, 
#                                    model="text-embedding-3-small")

# f = open('keys/.chroma_api_key.txt')
# CHROMA_API_KEY = f.read()


# # Initialize the database connection
# # If database exist, it will connect with the collection_name and persist_directory
# # Otherwise a new collection will be created
# vector_db = Chroma(collection_name="friends_collection", 
#             embedding_function=embedding_model, 
#             database='chroma_vector_db',
#             tenant="e5e7b9b9-292d-42b7-8379-d895e845610c",
#             chroma_cloud_api_key=CHROMA_API_KEY,
# )

In [14]:
from chromadb import Knn, K
from chromadb import Search
from chromadb import Rrf

def hybrid_retriever(query):
    # Sparse embeddings
    sparse_rank = Knn(
      query=query,  # Text query for sparse embeddings
      key="sparse_vector_key",  # Metadata field for sparse vectors
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)
    )
    
    # Dense semantic embeddings
    dense_rank = Knn(
      query=query,  # Text query for dense embeddings
      key="#embedding",          # Default embedding field
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)

    )
    
    # Combine with RRF
    hybrid_rank = Rrf(
      ranks=[dense_rank, sparse_rank],
      weights=[0.7, 0.3],  # 70% semantic, 30% keyword
      k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
    )
    
    # Use in search
    hybrid_search = (Search()
      .rank(hybrid_rank)
      .limit(5)
      .select(K.DOCUMENT, K.SCORE, K.METADATA)
    )
    
    hybrid_results = collection.search(hybrid_search)
    
    return hybrid_results

In [15]:
hybrid_retriever("Who is Rachem?")

{'ids': [['5f0d8817-5a47-4aaa-88cd-0993b5d5570e',
   'df79fc76-7da4-4846-8e5f-0d65c49a1900']],
 'documents': [['250\n00:15:17,423 --> 00:15:18,822\nWhat the hell\'s a Rachem?\n\n251\n00:15:19,125 --> 00:15:21,150\nIs that a stupid paleontology word...\n\n252\n00:15:21,394 --> 00:15:23,726\n... I wouldn\'t know,\nbecause I\'m just a waitress?\n\n253\n00:15:24,297 --> 00:15:25,787\nRach, come on!\n\n254\n00:15:28,001 --> 00:15:29,832\nIt\'s "She\'s not Rachel"!\n\n255\n00:15:30,103 --> 00:15:31,934\nShe\'s not....\n\n256\n00:15:39,913 --> 00:15:41,278\nMy diary! Brilliant!',
   '242\n00:14:53,433 --> 00:14:55,264\n"Just a waitress"?\n\n243\n00:14:56,569 --> 00:14:58,298\nNow that was....\n\n244\n00:14:58,738 --> 00:15:00,865\nI mean, as opposed to....\n\n245\n00:15:02,375 --> 00:15:04,707\nOkay, is this over yet? Rach?\n\n246\n00:15:05,478 --> 00:15:08,879\nI do not have chubby ankles!\n\n247\n00:15:09,082 --> 00:15:10,242\nNo! I\n\n248\n00:15:10,483 --> 00:15:13,611\nOkay, look at the o

In [16]:
# Step 3: Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

# Step 4: Initialize a Generator (i.e. Chat Model)

from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(api_key=OPENAI_API_KEY)

# Step 5: Initialize a Output Parser

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

generator_chain = prompt_template | chat_model | parser

In [19]:
# Helper function to join the retrieved chunks

def format_docs(docs):
    context = ""
    for row in docs.rows()[0]:
        context += row["document"]
    return context

In [20]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Wrap the functions in RunnableWrapper
hybrid_retriever_runnable = RunnableLambda(lambda x: hybrid_retriever(x))
format_docs_runnable = RunnableLambda(lambda x: format_docs(x))

In [21]:
rag_chain = {
    "context": hybrid_retriever_runnable | format_docs_runnable, 
    "question": RunnablePassthrough()
} | generator_chain

In [22]:
query = 'Who is Rachem?'

rag_chain.invoke(query)

'Rachem is a mistaken or mispronounced version of the name Rachel, as seen in the dialogue where someone insists that it\'s "She\'s not Rachel" and corrects the pronouncement of the name.'

In [27]:
query = 'What is there on the List comparing Rachel and Julie?'

rag_chain.invoke(query)

'The list compares the pros and cons of Rachel and Julie, as mentioned in the context.'